# 01 · Stencils and boundary conditions

The spatial half of the method of lines is a choice of stencils: which neighbouring values approximate a
derivative, with what order of accuracy, and what happens at the ends of the grid. This notebook derives stencil
weights for any derivative on any points (Fornberg's algorithm), measures their orders including the one-sided
closures at the boundaries, and then treats the boundary conditions themselves by ghost points: Dirichlet,
Neumann, Robin and periodic, on node-centred and cell-centred grids, with the test that every ghost formula
must pass.

**What you will learn**
- derive finite-difference weights on any points with Fornberg's algorithm
- measure the order of a differentiation operator including its boundary closures
- impose Dirichlet, Neumann, Robin and periodic conditions by ghost points on node and cell grids
- use consistency checks and stretched grids

**Before you start:** notebook 00  ·  **Time:** about 60 min

In [1]:
try:                      # on Google Colab (or anywhere engmolines is missing): install it from GitHub
    import engmolines
except ImportError:
    import subprocess, sys
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "git+https://github.com/ktwyw/engmolines"], check=True)
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from engmolines import grids, stencils as st, boundary as bc, operators as op, analysis, linalg as la
plt.rcParams.update({"figure.figsize": (7, 4), "figure.dpi": 90, "axes.grid": True, "grid.alpha": 0.3,
                     "axes.spines.top": False, "axes.spines.right": False})

import inspect
from IPython.display import Code

def show_source(obj):
    """Display the source code of a library function or class."""
    return Code(inspect.getsource(obj), language="python")

def heading(text):
    print(f"\n{text}\n" + "=" * len(text))

## Fornberg's weights

In [2]:
print("u_xx, 3 points:        ", st.fd_weights(0.0, [-1, 0, 1], 2))
print("u_xx, 5 points (4th):   ", st.fd_weights(0.0, [-2, -1, 0, 1, 2], 2) * 12, "/ 12")
print("u_x one-sided (2nd):    ", st.fd_weights(0.0, [0, 1, 2], 1) * 2, "/ 2")
print("u_x, non-uniform points:", np.round(st.fd_weights(0.3, [0.0, 0.5, 1.2], 1), 4))
print("u_xxxx, 5 points:       ", st.fd_weights(0.0, [-2, -1, 0, 1, 2], 4))

u_xx, 3 points:         [ 1. -2.  1.]
u_xx, 5 points (4th):    [ -1.  16. -30.  16.  -1.] / 12
u_x one-sided (2nd):     [-3.  4. -1.] / 2
u_x, non-uniform points: [-1.8333  1.7143  0.119 ]
u_xxxx, 5 points:        [ 1. -4.  6. -4.  1.]


Fornberg's recurrence gives the weights of the formula Σ w_j f(x_j) ≈ f^(m)(x₀) that is exact for polynomials
of the highest possible degree, on any set of points - centred or one-sided, uniform or not. Every stencil in
the library comes from it, including the biharmonic [1, −4, 6, −4, 1]/Δx⁴ for beams and the one-sided
formulas the boundaries need.

## Differentiation operators and their orders

In [3]:
f = lambda x: np.sin(3 * x)                                  # noqa: E731
exact = {1: lambda x: 3 * np.cos(3 * x), 2: lambda x: -9 * np.sin(3 * x), 4: lambda x: 81 * np.sin(3 * x)}
ns = [20, 40, 80, 160]
for deriv, acc in ((1, 2), (1, 4), (2, 2), (2, 4), (2, 6), (4, 2)):
    errs = [np.max(np.abs(st.diff_operator(np.linspace(0, 1, n + 1), deriv, acc)(f(np.linspace(0, 1, n + 1))) - exact[deriv](np.linspace(0, 1, n + 1)))) for n in ns]
    print(f"derivative {deriv}, accuracy {acc}: errors {['%.1e' % e for e in errs]}, observed orders {np.round(analysis.observed_order(errs), 2)}")
D = st.diff_operator(np.linspace(0, 1, 9), 2, 2)
print("\nthe 3-point u_xx operator on 9 points, as a dense matrix (times dx^2):\n", np.round(D.to_dense() * (1 / 8) ** 2, 3))

derivative 1, accuracy 2: errors ['2.2e-02', '5.6e-03', '1.4e-03', '3.5e-04'], observed orders [1.99 2.   2.  ]
derivative 1, accuracy 4: errors ['2.9e-04', '1.9e-05', '1.2e-06', '7.4e-08'], observed orders [3.96 3.99 4.  ]
derivative 2, accuracy 2: errors ['5.6e-02', '1.0e-02', '2.1e-03', '4.7e-04'], observed orders [2.44 2.29 2.17]
derivative 2, accuracy 4: errors ['1.5e-03', '6.3e-05', '3.0e-06', '1.5e-07'], observed orders [4.57 4.43 4.28]
derivative 2, accuracy 6: errors ['3.7e-05', '3.8e-07', '4.0e-09', '4.6e-10'], observed orders [6.63 6.55 3.12]
derivative 4, accuracy 2: errors ['2.0e+00', '3.5e-01', '6.7e-02', '1.4e-02'], observed orders [2.54 2.39 2.25]

the 3-point u_xx operator on 9 points, as a dense matrix (times dx^2):
 [[ 2. -5.  4. -1.  0.  0.  0.  0.  0.]
 [ 1. -2.  1.  0.  0.  0.  0.  0.  0.]
 [ 0.  1. -2.  1.  0.  0.  0.  0.  0.]
 [ 0.  0.  1. -2.  1.  0.  0.  0.  0.]
 [ 0.  0.  0.  1. -2.  1.  0.  0.  0.]
 [ 0.  0.  0.  0.  1. -2.  1.  0.  0.]
 [ 0.  0.  0.  0.  0.

`diff_operator` builds a differentiation operator of any derivative and accuracy: centred stencils inside,
one-sided stencils of the *same* accuracy at the ends (one more point each, so the matrix's first and last
rows reach further in). The orders come out as designed - and sometimes better, because the one-sided closures
happen to be more accurate for this function; the sixth-order operator reaches round-off (10⁻¹⁰, the
subtraction of nearly equal values divided by Δx²) on the finest grid, where its order collapses. The
operators are stored by diagonals and applied in O(n); a dense matrix is only formed for display or for
eigenvalues.

## Ghost points: the method-of-lines way to impose boundary conditions

In [4]:
u = np.array([1.0, 1.4, 1.9, 2.5, 3.2]); dx = 0.1
for name, left, right, kind in (("Dirichlet / Neumann, node grid", bc.Dirichlet(0.7), bc.Neumann(-2.0), "node"),
                                ("Robin / Neumann, cell grid", bc.Robin(2.0, 0.5, 3.0), bc.Neumann(1.5), "cell"),
                                ("periodic", bc.Periodic(), bc.Periodic(), "cell")):
    p = bc.pad(u, dx, left, right, kind=kind, nghost=2)
    print(f"{name:<32}: {np.round(p, 3)}")
p = bc.pad(u, dx, bc.Robin(2.0, 0.5, 3.0), bc.Neumann(1.5), kind="cell")
print(f"\nRobin check at the left face: alpha u + beta u_x = {2.0 * (p[0] + p[1]) / 2 + 0.5 * (p[1] - p[0]) / dx:.3f} (should be 3)")
p = bc.pad(u, dx, bc.Dirichlet(0.7), bc.Neumann(-2.0), kind="node")
print(f"Neumann check at the right node: (u_ghost - u_inner)/(2 dx) = {(p[-1] - p[-3]) / (2 * dx):.3f} (should be -2)")

Dirichlet / Neumann, node grid  : [-0.5  0.   1.   1.4  1.9  2.5  3.2  2.1  1.1]
Robin / Neumann, cell grid      : [0.65 0.75 1.   1.4  1.9  2.5  3.2  3.35 2.95]
periodic                        : [2.5 3.2 1.  1.4 1.9 2.5 3.2 1.  1.4]

Robin check at the left face: alpha u + beta u_x = 3.000 (should be 3)
Neumann check at the right node: (u_ghost - u_inner)/(2 dx) = -2.000 (should be -2)


Rather than writing special stencils for the boundary rows, the state is extended by ghost values chosen so
that the boundary condition holds (to second order), and the interior stencil is applied everywhere. A
Dirichlet value is imposed by odd reflection about the boundary, a Neumann slope by even reflection plus the
slope, a Robin condition by solving the small equation that links boundary value and slope; periodic grids
simply wrap. On a node grid the boundary point is itself a grid point - its Dirichlet value is fixed and its
rate set to zero (or to dg/dt for a time-dependent value) - while a cell-centred grid has no point on the
boundary and imposes everything through the ghosts. The checks above are the tests every ghost formula must
pass: evaluate the condition with the ghosts in place and get the imposed value back.

## Consistency: what the operator does to a polynomial

In [5]:
g = grids.uniform(0, 1, 40)
print("u = x^2, Dirichlet ends: u_xx =", np.unique(np.round(op.diffusion(g.x**2, g, 1.0, bc.Dirichlet(0.0), bc.Dirichlet(1.0))[1:-1], 10)))
gc = grids.cell_centred(0, 1, 40)
k = np.where(gc.x < 0.5, 1.0, 4.0)
u_layer = np.where(gc.x < 0.5, gc.x, 0.5 + 0.25 * (gc.x - 0.5))          # continuous flux k u_x = 1 across the interface at the face x = 0.5
print("composite wall, flux-continuous profile: max |residual| =", f"{np.max(np.abs(op.diffusion(u_layer, gc, k, bc.Dirichlet(0.0), bc.Dirichlet(0.625)))):.1e}")
gr = grids.radial(1.0, 40, "cylindrical")
print("cylindrical (1/r)(r u_r)_r of r^2 (should be 4, including r = 0):", np.unique(np.round(op.diffusion(gr.x**2, gr, 1.0, None, bc.Dirichlet(1.0))[:-1], 8)))

u = x^2, Dirichlet ends: u_xx = [2.]
composite wall, flux-continuous profile: max |residual| = 7.1e-13
cylindrical (1/r)(r u_r)_r of r^2 (should be 4, including r = 0): [4.]


Three consistency checks that catch most discretisation errors: a quadratic must give the exact second
derivative; a piecewise-linear profile with continuous flux through a conductivity jump must be a steady state
(the operator averages conductivities harmonically at the faces - which is why material interfaces belong on
faces, so the composite wall lives on a cell-centred grid); and the r = 0 axis of a cylindrical grid must be
handled by a control volume, not by dividing by zero.

## Non-uniform grids

In [6]:
gs = grids.stretched(0, 1, 40, cluster=0.5, strength=2.0)
print(f"stretched grid: spacing from {np.diff(gs.x).min():.4f} to {np.diff(gs.x).max():.4f}; the differentiation operator uses Fornberg weights at every point")
f2 = lambda x: np.exp(-((x - 0.5) / 0.05) ** 2)                 # noqa: E731, a narrow bump at the cluster point
d2 = lambda x: f2(x) * (4 * (x - 0.5) ** 2 / 0.05**4 - 2 / 0.05**2)   # noqa: E731
gu = grids.uniform(0, 1, 40)
print(f"u_xx of the bump: error on the uniform grid {np.max(np.abs(st.diff_operator(gu.x, 2, 2)(f2(gu.x)) - d2(gu.x))):.2e}, on the stretched grid {np.max(np.abs(st.diff_operator(gs.x, 2, 2)(f2(gs.x)) - d2(gs.x))):.2e}")

stretched grid: spacing from 0.0121 to 0.1094; the differentiation operator uses Fornberg weights at every point
u_xx of the bump: error on the uniform grid 9.22e+01, on the stretched grid 2.28e+01


The same 40 points, clustered where the solution varies, reduce the error of the second derivative of a narrow
bump by a factor of four (the bump is still under-resolved; the gain grows with the clustering). Stretched grids cost nothing in the method of lines - the stencil weights absorb
the spacing - and are the usual remedy for boundary layers and fronts of known position.

## Inside the algorithm: a Neumann ghost by hand

In [7]:
u_in = np.array([1.0, 1.4, 1.9]); slope = -2.0; dx = 0.1
ghost = u_in[1] - 2 * dx * slope                   # even reflection about the end node, plus the slope
print("ghost by hand:", ghost, "; library:", bc.pad(u_in, dx, bc.Neumann(slope), bc.Neumann(0.0), kind="node")[0])
print("centred slope at the end node:", (u_in[1] - ghost) / (2 * dx))

ghost by hand: 1.7999999999999998 ; library: 1.7999999999999998
centred slope at the end node: -1.9999999999999996


## Exercises
1. Build a fourth-order Laplacian with fourth-order one-sided closures and a fourth-order Laplacian with only second-order closures (use `fd_weights` on 4 points at the ends). Measure both orders. Which closure order does the global error follow, and why?
2. On a cell-centred grid impose u = 1 at x = 0 by the ghost formula and compare the resulting boundary flux k u_x with the exact one for the steady solution of u_xx = 0, u(0) = 1, u(1) = 0. What order is the flux error?
3. **Implement it yourself:** write Fornberg's algorithm (or solve the Vandermonde system for the weights) for the first derivative on the points (0, 0.5, 1.2) at x = 0.3 and compare with `stencils.fd_weights`.